# From a function to a model: a Hippocampus workflow

What a researcher does with `ai4science-client`, start to finish:

1. **Run** a Python function on Snellius.
2. **Train** a model as a tracked job: the job logs to its own MLflow run and
   registers the model.
3. **Compare** a small parameter sweep.
4. **Use** the model on your laptop.
5. **Infer** on Snellius with the registered model.

All code that runs on the cluster is a plain function in this notebook; the
client ships it with the job. Nothing here talks to Slurm or MLflow
directly, and you only ever see your own jobs, runs and models.

This notebook runs against the **playground** server,
`https://ai4science.playground.sdp.surf.nl`, with your Snellius login.

**Before you start**, put your login in a `.env` file in the root of the
client repository (next to `.env.example`):

```bash
export AI4SCIENCE_BASE_URL="https://ai4science.playground.sdp.surf.nl"
export SLURM_USER="your_snellius_user"
export SLURM_TOKEN="..."
```

Get a token on a Snellius login node with `scontrol token lifespan=86400`
(valid for one day) and copy the part after `SLURM_JWT=`.

**Start Jupyter** from the root of the client repository. Jobs install the
same MLflow and scikit-learn versions on the cluster, so the model from
step 4 loads here:

```bash
uvx --python 3.12 --from jupyterlab \
    --with mlflow==3.18.0 --with scikit-learn==1.5.2 --with pandas==2.2.3 --with skops==0.16.0 \
    --with-editable . \
    jupyter-lab examples/user-workflow-playground.ipynb
```

## 0. Connect

The client reads the server address and your Snellius login from `.env`.
Every job asks for the same small CPU slot, and jobs that use MLflow install
the versions this notebook runs with (`ML_DEPS`).

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # the .env of this repository

from ai4science_client import Ai4ScienceClient, build_script
from ai4science_client.schemas import SlurmResourceConfig

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"  # quiets an MLflow log hint in step 4

# Reads AI4SCIENCE_BASE_URL, SLURM_USER and SLURM_TOKEN.
client = Ai4ScienceClient(interval=5)
CPU = SlurmResourceConfig(
    partition="genoa",
    cpus_per_task=2,
    memory_mb=4000,
    time_limit_minutes=15,
    tres_per_node="",
)
ML_DEPS = ["mlflow-skinny==3.18.0", "scikit-learn==1.5.2", "skops==0.16.0"]
print(f"connected to {client.base_url} as {client.user}")

Along the way the notebook links into the **MLflow UI**, where the same runs
and models show up with charts and files. It asks for a login: use the demo
account you were given. This helper prints a clickable link:

In [ ]:
from IPython.display import Markdown, display


def link(label: str, url: str) -> None:
    display(Markdown(f"**MLflow UI:** [{label}]({url})"))

## 1. Run a function on the cluster

`client.run` packs the function and its arguments into a job, waits, and
returns what the function returned. The function must be self-contained:
imports go inside it.

In [ ]:
def where_am_i(greeting):
    import os
    import platform

    return {
        "greeting": greeting,
        "host": platform.node(),
        "python": platform.python_version(),
        "cpus": os.cpu_count(),
    }


client.run(where_am_i, "hello from the notebook", runtime="apptainer", resources=CPU)

## 2. Train a model as a tracked job

With `track=True`, Hippocampus opens an MLflow run before the job starts and
gives the job what it needs to log to it: plain `mlflow.start_run()` picks up
that run, and `HIPPOCAMPUS_MODEL_PREFIX` is your own namespace for model
names. The job can only reach your runs and models.

In [ ]:
EXPERIMENT = "iris-notebook"


def train(C):
    import os

    import mlflow
    from sklearn.datasets import load_iris
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import f1_score
    from sklearn.model_selection import train_test_split

    X, y = load_iris(return_X_y=True, as_frame=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
    with mlflow.start_run():
        model = LogisticRegression(C=C, max_iter=500).fit(X_train, y_train)
        predicted = model.predict(X_test)
        metrics = {
            "accuracy": float((predicted == y_test).mean()),
            "f1_macro": float(f1_score(y_test, predicted, average="macro")),
        }
        mlflow.log_param("C", C)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(
            model,
            name="model",
            input_example=X_train.head(3),
            registered_model_name=os.environ["HIPPOCAMPUS_MODEL_PREFIX"] + "iris",
        )
    return metrics


job = client.submit(
    build_script(train, (1.0,)),
    runtime="apptainer",
    dependencies=ML_DEPS,
    resources=CPU,
    track=True,
    experiment_name=EXPERIMENT,
)
print("job", job.job_id, "-> slurm job", job.slurm_job_id)
print("result", job.wait().result)

The job's MLflow run, as the client reads it back (the link opens the MLflow
UI):

In [ ]:
run = job.tracking_run()
print("status ", run["status"])
print(
    "params ",
    {k: v for k, v in run["params"].items() if not k.startswith("resources.")},
)
print("metrics", {k: round(v, 4) for k, v in run["metrics"].items()})

MLFLOW_UI, _, run_path = run["ui_url"].partition("/#/")
EXPERIMENT_URL = f"{MLFLOW_UI}/#/{run_path.rsplit('/runs/', 1)[0]}"
link("this training run", run["ui_url"])

In the run page:

- **Overview**: the parameters (`C`, and the resources the job asked for) and
  the metrics the job logged.
- **Artifacts → model**: what the job saved. `MLmodel` says how to load it,
  `requirements.txt` what it needs. This is exactly what step 4 downloads.
- **Registered model**: the `iris` version this run created.

## 3. A small sweep

`submit` returns right away, so several jobs can run side by side. Every run
lands in the same experiment, and `client.runs` lists them.

In [ ]:
sweep = {
    C: client.submit(
        build_script(train, (C,)),
        runtime="apptainer",
        dependencies=ML_DEPS,
        resources=CPU,
        track=True,
        experiment_name=EXPERIMENT,
    )
    for C in (0.01, 0.1, 10.0)
}
for C, j in sweep.items():
    print(f"C={C:<5} accuracy {j.wait().result['accuracy']:.3f}")

In [ ]:
import pandas as pd

pd.DataFrame(
    {
        "run": r["run_id"][:8],
        "slurm job": r["job_id"],
        "status": r["status"],
        "started (UTC)": pd.to_datetime(r["start_time"], unit="ms").floor("s"),
        **{k: round(v, 3) for k, v in r["metrics"].items()},
    }
    for r in client.runs(EXPERIMENT)
)

In [ ]:
link(f"experiment {EXPERIMENT}, all runs", EXPERIMENT_URL)

In the experiment page, tick the sweep's runs and press **Compare**: a
parallel-coordinates plot of `C` against `accuracy` shows the trade-off at a
glance. The **Chart** view plots every run's metrics side by side.

## 4. Use the model on your laptop

Every training job above registered a new version of `iris`, and each
version knows the run that made it. Take the version from the sweep's best
run: you get a plain MLflow model directory that runs anywhere with the
libraries in its `requirements.txt`, without access to the cluster or MLflow.

In [ ]:
for m in client.list_models():
    versions = ", ".join(f"v{v['version']}" for v in m["versions"])
    print(f"{m['name']:8} {versions}")

In [ ]:
best_run = max(client.runs(EXPERIMENT), key=lambda r: r["metrics"].get("accuracy", 0))
iris = next(m for m in client.list_models() if m["name"] == "iris")
best = next(v["version"] for v in iris["versions"] if v["run_id"] == best_run["run_id"])
print(
    f"best run {best_run['run_id'][:8]} (accuracy {best_run['metrics']['accuracy']:.3f})"
    f" registered iris v{best}"
)
link("model iris, all versions", f"{MLFLOW_UI}/#/models/{iris['registered_name']}")
link(f"iris v{best}", f"{MLFLOW_UI}/#/models/{iris['registered_name']}/versions/{best}")

In [ ]:
import mlflow.pyfunc
from sklearn.datasets import load_iris

model_dir = client.download_model("iris", best, dest="downloads")
model = mlflow.pyfunc.load_model(str(model_dir))
X, y = load_iris(return_X_y=True, as_frame=True)
predicted = model.predict(X)
print(model_dir, f"accuracy on all 150 flowers: {(predicted == y).mean():.3f}")
X.head(3).assign(predicted=predicted[:3], actual=y[:3])

## 5. Batch inference on the cluster

The same model version, used by a job: it loads
`models:/<your prefix>iris/<version>`, scores the data on the cluster, and
logs the result to its own run.

In [ ]:
def score(version):
    import os

    import mlflow
    from sklearn.datasets import load_iris

    uri = f"models:/{os.environ['HIPPOCAMPUS_MODEL_PREFIX']}iris/{version}"
    X, y = load_iris(return_X_y=True, as_frame=True)
    with mlflow.start_run():
        predicted = mlflow.pyfunc.load_model(uri).predict(X)
        accuracy = float((predicted == y).mean())
        mlflow.set_tag("model_uri", uri)
        mlflow.log_metric("accuracy", accuracy)
    return {"model": uri, "accuracy": accuracy}


scoring = client.submit(
    build_script(score, (best,)),
    runtime="apptainer",
    dependencies=ML_DEPS,
    resources=CPU,
    track=True,
    experiment_name="iris-scoring",
)
print(scoring.wait().result)
link("this scoring run", scoring.tracking_run()["ui_url"])

The scoring run sits in its own experiment, `iris-scoring`, and its
`model_uri` tag names the exact model version it used: from any result you
can walk back to the model, and from the model to the training run.

## 6. Choosing where a job runs

Everything above went to the server's default cluster, Snellius. The server
can also pick a cluster for you from its compute catalog: `tier="auto"` takes
the smallest cluster and partition that fit the job, `cluster="<name>"` pins
one. Logins for other clusters go in `logins=` on the client:

```python
client = Ai4ScienceClient(logins={"other-cluster": (OTHER_USER, OTHER_TOKEN)})
client.run(where_am_i, "hi", runtime="apptainer", tier="auto")
client.run(where_am_i, "hi", runtime="apptainer", cluster="other-cluster")
```

A cluster is only a candidate if it runs the job's runtime and you can log in
there. The job request stays the same on every cluster; the server fills in
each cluster's layout (work directory, system image, GPU support).

## What this used

| Step | Client call |
| --- | --- |
| Run a function | `client.run(func, ..., runtime="apptainer")` |
| Tracked job, its run | `client.submit(build_script(...), track=True)`, `job.wait()`, `job.tracking_run()` |
| Compare runs | `client.runs(experiment)`, the MLflow UI's Compare view |
| Your models | `client.list_models()`, `client.download_model(name)` |
| Pick a cluster | `tier=` / `cluster=`, `logins=` on the client |